In [2]:
import numpy as np #library to manipulate arrays
import matplotlib.pyplot as plt #library to plot figures
import scipy.signal as sci #signal processing library
%matplotlib qt

In [3]:
coef=np.loadtxt("coefbpcours.csv",delimiter=',')
b=coef[0,:] # non recursif coef
a=coef[1,:] # recursif coef

# Exercise 1

the function y=sci.lfilter(b,a,x) implement the following linear filter :

$y(n)=\sum_{i=0}^{N} b_i x(n-i)- \sum_{j=1}^{D} a_j y(n-i)$

here N=D=9

the filter seems stable since the impulse response goes to zeros with time

This is confirmed by the plot of the poles and zeros on the Z-plane : the poles are inside the unit circle.

In the frequency domain, Y(f)=H(f)X(f). 

For an impulse signal, we have X(f)=1, then Y(f), the FT of the impulse response will be equal to H(f), the filter frequency response. 

In [7]:
"""
Compute the impulse response h(n).
Compute the frequency response. What kind of filter is it ?
What can you say about the filter stability ?
By using the function z,p,k=scipy.signal.tf2zpk(b,a),
 plot the poles and the zeros in the complex plane. States the filter stability.
"""


# create the impulse signal
M=1024 #number of samples for the simulation
imp=np.zeros(M) #impulse signal
imp[0]=1

#filter the impulse signal
h=sci.lfilter(b,a,imp) #h is the impulse response of this filter

#plot the impulse response in teh time domain
plt.figure()
plt.plot(h[:100])# here, just the first 100 samples. to have the full plot plt.plot(h)  
plt.xlabel("frequency")
plt.ylabel(" ")
plt.title("impulse response")

#frequency response of the filter
H=np.fft.fft(h)  # frequency response
k=np.arange(1024) #frequency index
f=k/M #normalized frequnecies 
Fe=16 #freq d'échantillonnage en MHz
freal=f*Fe #real frequencies

plt.figure()
plt.plot(f,10*np.log10(np.abs(H)**2)) # in dB, replace f by freal to get true frequencies
plt.xlabel("frequency")
plt.ylabel("psd (dB)")
plt.title("frequency response")


Text(0.5, 1.0, 'frequency response')

### filter stability


In [6]:
z,p,k=sci.tf2zpk(b,a) # retrieve the zeros (z) and the poles (p) of the filter defined by the coefficients a and b
c=np.exp(2*np.pi*1j*np.linspace(0,1,1000)) # just for plotting the unit circle
plt.figure()
plt.plot(c.real,c.imag,label="unit circle") #the unit circle
plt.plot(z.real,z.imag,'o',label="zeros")
plt.plot(p.real,p.imag,'x',label="poles")
plt.legend()
plt.xlabel("real axis")
plt.ylabel("imaginary axis")
plt.title("Z-plane")

Text(0.5, 1.0, 'Z-plane')

# Exercise 2

In the frequency domain, Y(f)=H(f)X(f). 

For a white noise, we have X(f)=1, then Y(f), the FT of the impulse response will be equal to H(f), the filter frequency response. 

In [7]:
noise=np.random.randn(32768)
h=sci.lfilter(b,a,noise)

plt.figure()
plt.plot(h)
plt.xlabel("time index (n)")
plt.ylabel("amplitude ")
plt.title("white noise response")

plt.figure()
plt.psd(h,1024,Fs=1,label='Y(f)=H(f)')
plt.psd(noise,1024,Fs=1,label="X(f)=cst")
plt.xlabel("normalized frequency")
plt.ylabel("psd ")
plt.title("frequency response")
plt.legend()

# Exercise 3

$H(f)=\frac {FFT(b)}{FFT(a)}$

b and a can be extended by adding 0 (zero padding) to get :

1. a and b vector lengths equal to a power of 2
2. to get a sufficient frequency resolution

the function freqz from scipy.signal do the same directly.

In [19]:
M=512 #frequency number

#do it yourself approach
a_padding=np.zeros(M)
a_padding[:len(a)]=a
b_padding=np.zeros(M)
b_padding[:len(b)]=b
H=np.fft.fft(b_padding)/np.fft.fft(a_padding)
f=np.linspace(0,1,M)
plt.figure()
plt.plot(f,10*np.log10(np.abs(H)**2))
plt.xlabel("normalized frequency")
plt.ylabel("psd ")
plt.title("frequency response")
plt.title("do it yourself approach")

#the same but with freqz function
f,H=sci.freqz(b, a, worN=M, fs=1) #worN is the number of frequencies
plt.figure()
plt.plot(f,10*np.log10(np.abs(H)**2))
plt.xlabel("normalized frequency")
plt.ylabel("psd ")
plt.title("frequency response with freqz")

Text(0.5, 1.0, 'frequency response with freqz')

# exercise 4

the filtering of a sine wave with frequency fo gives  a sine wave with the same frequency fo but affected by a gain equal to H(fo).

Here, the idea is to generate a sine wave which will sweep all the frequencies over a given duration.

The filtered output should give the shape of H(f) over the given duration.  


In [20]:
nbs=32768
sweep=np.sin(np.pi*0.5/nbs*np.arange(nbs)**2) # a sweep frequency signal from f=0 to f=0.5
hsweep=sci.lfilter(b,a,sweep)
plt.figure()
plt.plot(sweep) #zoom it to see the sweeping
plt.xlabel("time index (n)")
plt.ylabel("amplitude ")
plt.title("sweep signal")
plt.figure()
plt.plot(np.abs(hsweep)) #the enveloppe of the filtered response gives the shape of the filter
plt.xlabel("time index (n)")
plt.ylabel("amplitude ")
plt.title("filtered sweep signal")

Text(0.5, 1.0, 'filtered sweep signal')

# exercise 5

In [ ]:
newb,newa=sci.zpk2tf(z,1/p, k) # we generate another set of coefficients b and a, from the previous pole and zeros

In [ ]:
#the frequency response seems exactly the same as the previous filter but...
f,H=sci.freqz(newb, newa, worN=512, fs=1)
plt.figure()
plt.plot(f,10*np.log10(np.abs(H)**2))
plt.xlabel("normalized frequency")
plt.ylabel("psd ")
plt.title("frequency response")

In [13]:
# the poles will be outside the unit circle => filter unstable
z,p,k=sci.tf2zpk(newb,newa)
c=np.exp(2*np.pi*1j*np.linspace(0,1,1000))
plt.figure()
plt.plot(c.real,c.imag)
plt.plot(z.real,z.imag,'o',label="zeros")
plt.plot(p.real,p.imag,'x',label="poles")

In [15]:
#if you plot the filter impulse response, you can see that it is diverging
imp=np.zeros(1024)
imp[0]=1
h=sci.lfilter(newb,newa,imp)
H=np.fft.fft(h)
plt.figure()
plt.plot(h)
plt.xlabel("time index (n)")
plt.ylabel("amplitude ")
plt.title("impulse response")

plt.figure()
plt.plot(10*np.log10(np.abs(H)**2))
plt.xlabel("frequency index k (k/M)")
plt.ylabel("psd ")
plt.title("frequency response")

Text(0.5, 1.0, 'frequency response')